# 🏨 Hotel Cancellation Prediction — Logistic Regression Lab
## Can we predict which bookings will be cancelled before they happen?

---

### The Business Problem

Hotel cancellations are a massive headache for the hospitality industry.  
Consider a hotel with 200 rooms that is fully booked for a Saturday night:

- If **20% of bookings cancel** (a realistic figure), the hotel loses revenue on 40 rooms it could have re-sold.
- If the hotel **overbooks aggressively** and fewer people cancel than expected, it must walk guests to a competitor — destroying customer loyalty.
- You are a data scientist at a fast growing hotel brand and they need a better way to predict cancellations, they have a pretty good size
dataset and would like you to help. If the model works well you get a 10% bonus of all the recovered revenue!  

**A good cancellation-prediction model lets hotels:**
1. **Optimise overbooking** — sell the right number of extra rooms so the hotel fills up even after cancellations.
2.  **Improve staffing** — if 30% of tonight's bookings will cancel, fewer housekeepers and front-desk staff are needed.
3.  **Target at-risk bookings** — reach out proactively with incentives (free breakfast, discount upgrade) to guests who are likely to cancel.

### The Dataset

We are working with the **Hotel Booking Demand** dataset (~119 k real bookings from two Portuguese hotels).  
Our **target variable** is `is_canceled` — `1` if the booking was cancelled, `0` if the guest actually showed up.

### What You Will Build

| Model | Key idea |
|-------|----------|
| **Model 1** | Baseline pipeline — default solver (`lbfgs`), one-hot encoding + standard scaling |
| **Model 2** | Solver exploration — try using different solvers and see if the model improves |
| **Model 3** | Class-weighted logistic regression + cross-validation to handle imbalanced labels |



---
## 2 · Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    classification_report
)

sns.set_style('whitegrid')

In [2]:
df = pd.read_csv('hotels.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

Dataset shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


---
## 3 · Exploratory Data Analysis

Before building any model we need to **understand our data**.  


In [ ]:
# Print out summary stats and basic info on the dataset


In [ ]:
# Calculate prevalence of the target value, how hard of problem is this going to be?

In [ ]:
# Missing value check for missing data and drop features with too many missing values

Columns with missing values:
children         4
country        488
agent        16340
company     112593
dtype: int64


In [ ]:
# There are two other features that need to be dropped because they  directly leak the target variable. Can you spot them? Drop them.

---
### Feature groups

We split our remaining features into **numerical** and **categorical** so we can apply the right transformations to each group inside a `ColumnTransformer`.

> **Why scale numerics?** Logistic regression uses gradient descent internally.  
> Features on very different scales (e.g. `lead_time` 0–737 vs `adults` 0–4) slow convergence.  
> `StandardScaler` fixes this by converting each feature to mean=0, std=1.

> **Why one-hot encode categoricals?** Logistic regression can't handle strings.  
> One-hot encoding converts each category into its own binary (0/1) column.

Keep in mind that we want to use the pipeline package to handle this preprocessing so create lists of names of numeric and cat then pass them into the pipeline. 



---

Build the ColumnTransformer (shared by all three models) make sure to do so for both **numerical** and **categorical**

---

Train / test split — 80/20, stratified so class ratios are preserved 

---
Model 1 — Baseline Logistic Regression

Our first pipeline is intentionally **simple**.  
The goal is to establish a baseline score we can try to beat later.

```
X_raw  →  ColumnTransformer  →  LogisticRegression(solver='lbfgs')
           (scale + encode)
```

**`lbfgs`** (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) is sklearn's default solver —  
it works well on medium-sized datasets and supports L2 regularisation. Use your pipe from above to fit the model

---
 Model 1 Evaluation

The Confusion Matrix — Your Most Important Evaluation Tool

For a hotel-cancellation problem the four cells of the confusion matrix have real business meaning:

| | Predicted: Kept | Predicted: Cancelled |
|---|---|---|
| **Actual: Kept** | ✅ **True Negative (TN)** — correctly identified loyal guest | ❌ **False Positive (FP)** — wrongly flagged a loyal guest as a canceller |
| **Actual: Cancelled** | ❌ **False Negative (FN)** — missed a cancellation (costly!) | ✅ **True Positive (TP)** — correctly predicted a cancellation |

**Business impact:**
- **False Negatives** are expensive — we didn't anticipate the cancellation, so we can't fill the room.
- **False Positives** are annoying but cheaper — we might over-staff or over-book slightly.

In [ ]:
y_pred_1 = model_1.predict(X_test)

print('=== Model 1 — Classification Report ===')
print(classification_report(y_test, y_pred_1, target_names=['Kept (0)', 'Cancelled (1)']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_1,
    display_labels=['Kept', 'Cancelled'],
    cmap='Blues', ax=ax
)
ax.set_title('Model 1 — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

acc_1 = (y_pred_1 == y_test).mean()
auc_1 = roc_auc_score(y_test, model_1.predict_proba(X_test)[:, 1])
print(f'Accuracy : {acc_1:.4f}')
print(f'ROC-AUC  : {auc_1:.4f}')

---
## 7 · Model 2 — Exploring Different Solvers & Regularisation

Sklearn's `LogisticRegression` supports several **solvers** (optimisation algorithms) and  
**regularisation strategies** controlled by `l1_ratio` and `C`.

| Solver | Best for |
|--------|----------|
| `lbfgs` | Default; works well on most datasets (L2 regularisation) |
| `liblinear` | Smaller datasets; supports L1 via `l1_ratio=1` |
| `saga` | Large datasets; supports L1, L2, and ElasticNet |

**Regularisation via `l1_ratio`** (sklearn 1.8+):
- **`l1_ratio=0`** (L2 / Ridge): shrinks all coefficients towards zero — keeps all features but small.
- **`l1_ratio=1`** (L1 / Lasso): can set some coefficients *exactly* to zero — acts as automatic feature selection.
- **`0 < l1_ratio < 1`** (ElasticNet): mix of both — only supported with the `saga` solver.

The **`C` parameter** controls regularisation strength: smaller C → stronger regularisation.

---

### 🎯 Student Exercise

In the cell below we use `liblinear` with L1-equivalent regularisation (`l1_ratio=1`).  
**Try changing these parameters** and see how accuracy and AUC change:
- `solver='saga'` with `l1_ratio=0.5` (ElasticNet)
- `C=0.1` vs `C=10.0`
- What happens to the number of zero coefficients when you use `l1_ratio=1` vs `l1_ratio=0`?

> 💡 **Hint:** After fitting, you can inspect `model_2['classifier'].coef_` to see the learned weights.

In [ ]:
# ── Model 2: liblinear solver + L1-equivalent regularisation ─────────────
# sklearn 1.8+: use l1_ratio=1 for pure L1, l1_ratio=0 for pure L2
# 🎯 Try changing: l1_ratio (0=L2, 1=L1), C (lower=stronger regularisation)
model_2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(
        solver='liblinear',   # 🎯 try: 'saga'
        l1_ratio=1,           # 🎯 try: 0 for L2, 0.5 for ElasticNet (saga only)
        C=1.0,                # 🎯 try: 0.1 or 10.0
        max_iter=1000,
        random_state=42
    ))
])

model_2.fit(X_train, y_train)
print('Model 2 trained ✅')

# How many coefficients are exactly zero? (L1 feature selection)
coef = model_2.named_steps['classifier'].coef_[0]
print(f'Total features after encoding : {len(coef)}')
print(f'Coefficients set to zero (L1) : {(coef == 0).sum()}')

---
## 8 · Model 2 Evaluation — ROC / AUC Curve

### What is the ROC Curve?

A logistic regression model outputs a **probability** (0–1) for each booking.  
We need to pick a **threshold** — e.g. 0.5 — to convert probabilities to a yes/no prediction.

The **ROC curve** plots every possible threshold:
- **X-axis**: False Positive Rate (how often we wrongly predict cancellation when the booking is kept)
- **Y-axis**: True Positive Rate (how often we correctly predict a cancellation)

**AUC** (Area Under the Curve) summarises this in a single number:
- **0.5** = random guessing (diagonal line)
- **1.0** = perfect classifier
- **> 0.8** = generally considered a good model

---

### 🎯 Student Exercise

After looking at the ROC curves below:
1. Which model performs better — Model 1 or Model 2?
2. If the hotel wants to **minimise missed cancellations** (False Negatives), should the threshold be higher or lower than 0.5?  
   *(Hint: a lower threshold flags more bookings as potential cancellations.)*

In [ ]:
y_prob_1 = model_1.predict_proba(X_test)[:, 1]
y_prob_2 = model_2.predict_proba(X_test)[:, 1]

fpr_1, tpr_1, _ = roc_curve(y_test, y_prob_1)
fpr_2, tpr_2, _ = roc_curve(y_test, y_prob_2)
auc_2 = roc_auc_score(y_test, y_prob_2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_1, tpr_1, label=f'Model 1 — lbfgs / L2  (AUC = {auc_1:.4f})', lw=2)
ax.plot(fpr_2, tpr_2, label=f'Model 2 — liblinear / L1 (AUC = {auc_2:.4f})', lw=2, linestyle='--')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random guess')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Model 1 vs Model 2', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Model 1 AUC: {auc_1:.4f}')
print(f'Model 2 AUC: {auc_2:.4f}')

In [ ]:
y_pred_2 = model_2.predict(X_test)
print('=== Model 2 — Classification Report ===')
print(classification_report(y_test, y_pred_2, target_names=['Kept (0)', 'Cancelled (1)']))

---
## 9 · Model 3 — Balanced Class Weights + Cross-Validation

### The Class Imbalance Problem

Our dataset has ~63% non-cancellations and ~37% cancellations.  
A lazy model could achieve 63% accuracy by *always* predicting "Kept"!  
We saw that Models 1 and 2 sometimes struggled with **recall on the cancelled class**.

### Solution: `class_weight='balanced'`

Setting `class_weight='balanced'` tells sklearn to automatically weight the training samples  
so that the minority class (cancellations) gets more attention during training.

Specifically, each class gets weight = `n_samples / (n_classes × n_samples_in_class)`.

### Solution: Cross-Validation

Instead of a single train/test split (which can be lucky or unlucky),  
**k-fold cross-validation** trains and evaluates the model on `k` different splits  
and reports the average — giving a much more reliable estimate of real-world performance.

```
Fold 1: [train | train | train | train | VAL ]
Fold 2: [train | train | train | VAL  | train]
Fold 3: [train | train | VAL  | train | train]
...and so on
```

---

### 🎯 Student Exercise

- Change the number of folds from `5` to `10` — does the mean AUC change much?
- Try removing `class_weight='balanced'` — what happens to recall on the cancelled class?

In [ ]:
# ── Model 3: balanced weights + cross-validation ─────────────────────────
# We go back to the reliable lbfgs solver but add class_weight='balanced'
# so the model pays more attention to the minority class (cancellations).
model_3 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(
        solver='lbfgs',
        C=1.0,
        class_weight='balanced',  # ← key change vs Models 1 & 2!
        max_iter=1000,
        random_state=42
    ))
])

# ── Cross-validation on the TRAINING set ──────────────────────────────────
# 🎯 Try changing n_splits to 10 — does the mean AUC change much?
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc_scores = cross_val_score(
    model_3, X_train, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f'Cross-validation AUC scores : {cv_auc_scores.round(4)}')
print(f'Mean CV AUC                 : {cv_auc_scores.mean():.4f} ± {cv_auc_scores.std():.4f}')

In [ ]:
# Fit on full training set for final evaluation
model_3.fit(X_train, y_train)
print('Model 3 fitted on full training set ✅')

---
## 10 · Model 3 — Full Evaluation

Let's do a comprehensive evaluation of our best model:  
**confusion matrix + classification report + ROC curve**, all together.

In [ ]:
y_pred_3 = model_3.predict(X_test)
y_prob_3 = model_3.predict_proba(X_test)[:, 1]

auc_3  = roc_auc_score(y_test, y_prob_3)
fpr_3, tpr_3, _ = roc_curve(y_test, y_prob_3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_3,
    display_labels=['Kept', 'Cancelled'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Model 3 — Confusion Matrix', fontsize=13)

# Right: ROC comparison of all three models
axes[1].plot(fpr_1, tpr_1, label=f'Model 1 lbfgs/L2    AUC={auc_1:.4f}', lw=2)
axes[1].plot(fpr_2, tpr_2, label=f'Model 2 liblinear/L1 AUC={auc_2:.4f}', lw=2, ls='--')
axes[1].plot(fpr_3, tpr_3, label=f'Model 3 balanced    AUC={auc_3:.4f}', lw=2, ls='-.')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curves — All Three Models', fontsize=13)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('=== Model 3 — Classification Report ===')
print(classification_report(y_test, y_pred_3, target_names=['Kept (0)', 'Cancelled (1)']))

In [ ]:
# ── Final model comparison summary ───────────────────────────────────────
acc_2 = (y_pred_2 == y_test).mean()
acc_3 = (y_pred_3 == y_test).mean()

print(f'{"Model":<10} {"Accuracy":>10} {"ROC-AUC":>10}')
print('-' * 32)
print(f'{"Model 1":<10} {acc_1:>10.4f} {auc_1:>10.4f}')
print(f'{"Model 2":<10} {acc_2:>10.4f} {auc_2:>10.4f}')
print(f'{"Model 3":<10} {acc_3:>10.4f} {auc_3:>10.4f}')

---
## 11 · Summary & Reflection

### What We Learned

🏆 **Model Performance**
- All three models achieved AUC > 0.85 — well above random guessing (0.5).
- `class_weight='balanced'` (Model 3) improves recall on the **cancelled class** at the cost of slightly lower overall accuracy — often the right trade-off in a business context.
- Cross-validation gave us confidence that our results weren't just a lucky train/test split.

🔑 **Key Takeaways**

1. **Data leakage kills models in production** — always check whether your features could only exist *after* the event you're predicting.
2. **sklearn Pipelines are your best friend** — they bundle preprocessing + modelling into one object, preventing data leakage from fitting the scaler on the test set.
3. **Different solvers = different trade-offs** — `liblinear` with L1 can zero out irrelevant features; `saga` scales to large datasets.
4. **AUC > Accuracy** for imbalanced problems — a model that predicts everything as 'Not Cancelled' can have high accuracy but terrible AUC.

### 🎯 Challenge Extensions

If you want to push further:

1. **Tune `C`** using `GridSearchCV` over the range `[0.001, 0.01, 0.1, 1, 10, 100]`.
2. **Engineer new features** — e.g. `total_nights = stays_in_weekend_nights + stays_in_week_nights`.
3. **Try a different model** — replace `LogisticRegression` in the pipeline with `RandomForestClassifier` or `GradientBoostingClassifier`. Does AUC improve?
4. **Plot a precision-recall curve** — sometimes more informative than ROC when classes are very imbalanced.
5. **Interpret coefficients** — which features does Model 3 consider most predictive of cancellation?

---

Great work completing this lab! 🎉  
You've built a real machine-learning pipeline on a real-world dataset — the same kind of work data scientists do every day in the hospitality industry.